# Fast sweeping in DMLab
The policy builds a geodesic distance field and steers toward its waypoint.

In [ ]:
import sys
sys.path.insert(0, "..")  # run from notebooks/

from dataclasses import dataclass

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Arc, Polygon, Rectangle
from scipy.ndimage import distance_transform_edt, map_coordinates

from src.envs.dmlab_autonomous_exploration import compute_distance_map, compute_path_smooth, get_exploration_targets, replay_sweeps, segment_from_numpy
from src.envs.dmlab_autonomous_exploration.distance_map import _initial_fields, _solve_eikonal_2d
from src.tools.render_utils import (ACCENT_BLUE, ACCENT_GREEN, ACCENT_RED, HEADING_CONE, HEADING_CONE_EDGE, TEXT_LABEL, TEXT_PRIMARY,
                                    TEXT_SECONDARY, TRAIL_SHADOW)

# ── Config ───────────────────────────────────────────
@dataclass
class Config:
    map_path: str = "../data/dmlab_topdown_maps/100001269.png"
    waypoint: int = 0     # active waypoint index
    radius: int = 5       # controller search window, in cells
    reach: float = 26.0   # waypoint reached when d(x_t) < reach
    size: int = 512       # field display resolution

cfg = Config()

def rgb(color):
    """0-255 color tuple -> matplotlib float RGB."""
    return np.array(color[:3]) / 255

MAP_WALL, MAP_FREE = rgb((18, 20, 25)), rgb((51, 57, 67))
plt.rcParams.update({"figure.facecolor": "black", "axes.facecolor": "black", "savefig.facecolor": "black", "font.size": 9,
                     "text.color": rgb(TEXT_SECONDARY), "axes.edgecolor": rgb(TEXT_LABEL), "axes.labelcolor": rgb(TEXT_LABEL),
                     "xtick.color": rgb(TEXT_LABEL), "ytick.color": rgb(TEXT_LABEL)})

# ── Inputs / Data ────────────────────────────────────
image = cv2.cvtColor(cv2.imread(cfg.map_path), cv2.COLOR_BGR2RGB)  # [H, W, 3]
free, _ = segment_from_numpy(image)                                  # [H, W], 1 = walkable
height, width = free.shape
waypoints = get_exploration_targets(free)                            # (row, column): center and four corners
goal = waypoints[cfg.waypoint]
dim_map = (image * 0.4).astype(np.uint8)
wall = cv2.resize(free, (cfg.size, cfg.size), interpolation=cv2.INTER_NEAREST) == 0

# ── Plot helpers ─────────────────────────────────────
EXTENT = (-0.5, width - 0.5, height - 0.5, -0.5)  # data coordinates are (column, row) grid cells

def panels(columns, rows=1, size=3.3):
    """Black figure with a grid of borderless axes."""
    fig, axes = plt.subplots(rows, columns, figsize=(size * columns, size * rows + 0.3), squeeze=False)
    fig.subplots_adjust(left=0.04, right=0.99, wspace=0.03, hspace=0.2)
    for ax in axes.flat: ax.axis("off")
    return fig, axes

def show(ax, picture, title="", extent=EXTENT, **kwargs):
    ax.imshow(picture, extent=extent, **kwargs)
    ax.autoscale(False)  # markers and routes never rescale the panel
    ax.set_title(title)

def dot(ax, point, color, size=70, hollow=False):
    """Viewer marker at a (row, column) cell: filled with a white ring, or a hollow ring for upcoming waypoints."""
    ax.scatter(point[1], point[0], s=size, facecolors="none" if hollow else [rgb(color)], edgecolors=[rgb(color if hollow else TEXT_PRIMARY)],
               linewidths=1.5, zorder=6)

def route(ax, path, color=ACCENT_BLUE, lw=1.8):
    """Polyline of (row, column) points over the viewer's dark underlay."""
    ax.plot(path[:, 1], path[:, 0], color=rgb(TRAIL_SHADOW), lw=lw + 1.7, zorder=3, solid_capstyle="round")
    ax.plot(path[:, 1], path[:, 0], color=rgb(color), lw=lw, zorder=4, solid_capstyle="round")

def heading_cone(ax, position, heading):
    """Viewer heading cone at an (x, y) position facing the unit (x, y) vector `heading`."""
    side, front = np.array([-heading[1], heading[0]]), position + 6 * heading
    ax.add_patch(Polygon([position, front + 2 * side, front - 2 * side], zorder=5, fc=(*rgb(HEADING_CONE), HEADING_CONE[3] / 255),
                         ec=(*rgb(HEADING_CONE_EDGE), HEADING_CONE_EDGE[3] / 255)))

def fill(field):
    """Fill nonfinite travel times [H, W] from the nearest finite cell before interpolation."""
    rows, columns = distance_transform_edt(~np.isfinite(field), return_distances=False, return_indices=True)
    return field[rows, columns]

def field_image(field, d_max):
    """Travel time [H, W] -> RGB [S, S, 3]: Viridis with yellow near the waypoint, gray where unreached, dark walls."""
    up = lambda array, mode: cv2.resize(array, (cfg.size, cfg.size), interpolation=mode)
    reached = up(np.isfinite(field).astype(np.uint8), cv2.INTER_NEAREST) > 0
    value = up(fill(field).astype(np.float32), cv2.INTER_LINEAR)  # [S, S] smooth travel time
    picture = plt.get_cmap("viridis")((1 - value / d_max).clip(0, 1))[..., :3]
    picture[~reached], picture[wall] = MAP_FREE, MAP_WALL
    return picture

def heat(values):
    """Walkable-cell values [H, W] -> Viridis RGB with dark walls."""
    picture = plt.get_cmap("viridis")(values / values.max())[..., :3]
    picture[free == 0] = MAP_WALL
    return picture

def contours(ax, field, levels, color="white", **kwargs):
    ax.contour(np.ma.masked_where(free == 0, fill(field)), levels=levels, colors=[color], **kwargs)

## 1 · Map
The top-down RGB map is segmented into walkable cells. Waypoints are the walkable cells nearest the center and the four corners; the first is active.

In [ ]:
fig, axes = panels(2)
show(axes[0, 0], dim_map, "top-down map")
show(axes[0, 1], np.where(free[..., None] > 0, MAP_FREE, MAP_WALL), "walkable mask")
for ax in axes[0]:
    for i, point in enumerate(waypoints):
        dot(ax, point, ACCENT_RED, hollow=i != cfg.waypoint)
        if ax is axes[0, 0]: ax.text(point[1] + 4, point[0] - 3, str(i + 1), color=rgb(TEXT_PRIMARY))

## 2 · Speed
Wall-aware speed: $F(x)=1/(1+5\,e^{-w(x)/8})$, with $w$ the distance to the nearest wall.
The travel time $d$ solves $|\nabla d|=1/F$ with $d=0$ at the waypoint.

In [ ]:
distance0, speed = _initial_fields(free, goal, 5.0, 8)  # [H, W]: zero at the waypoint and infinite elsewhere; wall-aware speed
fig, axes = panels(3)
show(axes[0, 0], heat(distance_transform_edt(free)), "wall distance w")
show(axes[0, 1], heat(speed), "speed F")
show(axes[0, 2], np.where(free[..., None] > 0, MAP_FREE, MAP_WALL), "d: only the waypoint is known")
dot(axes[0, 2], goal, ACCENT_RED)

## 3 · Local update
A cell takes the smaller neighbour on each axis, $a$ (vertical) and $b$ (horizontal), and solves the eikonal equation locally:

$$d=\tfrac12\Big(a+b+\sqrt{2/F^2-(a-b)^2}\Big)\ \text{ if } |a-b|<1/F,\qquad d=\min(a,b)+1/F\ \text{ otherwise.}$$

It keeps the smaller of $d$ and its current value.

In [ ]:
# up, down, left, right neighbour values around a cell with F = 1
cases = {"|a − b| < 1/F": (1.0, 2.2, 1.4, 3.0), "|a − b| ≥ 1/F": (1.0, 2.2, 3.0, 4.0)}
fig, axes = panels(2, size=3.0)
for ax, (title, (up, down, left, right)) in zip(axes[0], cases.items()):
    a, b = min(up, down), min(left, right)
    new = _solve_eikonal_2d(a, b, 1.0)
    for (r, c), value, label in (((0, 1), up, "a"), ((2, 1), down, ""), ((1, 0), left, "b"), ((1, 2), right, ""), ((1, 1), new, "d")):
        color = rgb(TEXT_PRIMARY if label == "d" else ACCENT_BLUE if value in (a, b) else TEXT_LABEL)
        ax.add_patch(Rectangle((c - 0.45, r - 0.45), 0.9, 0.9, fill=False, ec=color, lw=1.5))
        ax.text(c, r, f"{label} = {value:.2f}" if label else f"{value:.2f}", ha="center", va="center", color=color)
    ax.set_xlim(-0.6, 2.6); ax.set_ylim(2.6, -0.6); ax.set_aspect("equal"); ax.set_title(title)

## 4 · Sweeps
Each Gauss-Seidel sweep immediately reuses updated neighbor values.
Four traversal orders propagate information across the grid.

In [ ]:
# 1) Replay the solver: one frame per sweep, and in 16-row blocks for the first sweep
sweeps = [(d.copy(), step) for d, _, step, _ in replay_sweeps(free, goal, height)]  # 16 x [H, W]: 4 iterations x 4 directions
blocks = [d.copy() for d, *_ in replay_sweeps(free, goal, 16)]                      # height / 16 blocks per sweep
d_max = sweeps[-1][0][np.isfinite(sweeps[-1][0])].max()                             # fixed color scale

def field_panel(ax, field, title="", d_max=d_max, point=goal):
    show(ax, field_image(field, d_max), title)
    dot(ax, point, ACCENT_RED, 45)

def direction(step):
    return f"rows {'↓' if step[0] > 0 else '↑'}  columns {'→' if step[1] > 0 else '←'}"

# 2) First sweep row by row (from the waypoint's row on), then the four sweeps of iteration 1
fig, axes = panels(4, rows=2)
for ax, i in zip(axes[0], np.linspace(goal[0] // 16, height // 16 - 1, 4).astype(int)):
    field_panel(ax, blocks[i], f"rows ≤ {16 * (i + 1)}")
for ax, (field, step) in zip(axes[1], sweeps[:4]):
    field_panel(ax, field, direction(step))
for ax, label in zip(axes[:, 0], ("sweep 1, by rows", "iteration 1, by sweeps")):
    ax.text(-0.04, 0.5, label, transform=ax.transAxes, rotation=90, ha="right", va="center")

## 5 · Convergence
Four iterations of four sweeps; the plot reports the percentage of walkable cells updated at each sweep.

In [ ]:
fields = [distance0] + [field for field, _ in sweeps]
changed = [100 * (new != old).sum() / free.sum() for old, new in zip(fields[:-1], fields[1:])]  # % of walkable cells updated per sweep

fig = plt.figure(figsize=(13.2, 5.6))
grid = fig.add_gridspec(2, 4, height_ratios=[3, 1.2], hspace=0.25, wspace=0.03)
for i in range(4):
    ax = fig.add_subplot(grid[0, i]); ax.axis("off")
    field_panel(ax, sweeps[4 * i + 3][0], f"iteration {i + 1}")
ax = fig.add_subplot(grid[1, :])
ax.bar(range(1, 17), changed, color=rgb(ACCENT_BLUE), width=0.7)
for x in (4.5, 8.5, 12.5): ax.axvline(x, color=rgb(TEXT_LABEL), lw=0.6)
ax.set_xticks(range(1, 17)); ax.set_xlabel("sweep"); ax.set_ylabel("% cells updated")
ax.spines[["top", "right"]].set_visible(False)

## 6 · Descent
Routes follow $-\nabla d$ toward the waypoint. The controller changes target when $d<26$.

In [ ]:
field, _ = compute_distance_map(free, goal)                                                   # [H, W] float32, infinite at walls
terrain = fill(field)                                                                          # [H, W] finite travel time for surfaces

def descend(point):
    """Gradient-descent route [N, 2] (row, column) and its d [N], cut where the agent would switch waypoint."""
    route_ = compute_path_smooth(tuple(point), field)
    d = map_coordinates(terrain, route_.T, order=1)
    stop = np.argmax(d < cfg.reach) + 1
    return route_[:stop], d[:stop]

start = np.unravel_index(np.where(np.isfinite(field), field, -1).argmax(), field.shape)        # farthest reachable cell
path, height_along = descend(start)
reached = np.argwhere(np.isfinite(field))
starts = reached[np.linspace(0, len(reached) - 1, 14).astype(int)]                             # cells spread over the maze

fig = plt.figure(figsize=(14, 4.4))
grid = fig.add_gridspec(1, 3, wspace=0.08, left=0.01, right=0.99)

# 1) Topographic view: iso-distance lines every 20 cost units, route crossing them
ax = fig.add_subplot(grid[0, 0]); ax.axis("off")
field_panel(ax, field, "d as terrain")
contours(ax, field, np.arange(0, d_max, 20), linewidths=0.7, alpha=0.55)
route(ax, path); dot(ax, start, ACCENT_GREEN); dot(ax, goal, ACCENT_RED)

# 2) Routes from many cells converge on the waypoint
ax = fig.add_subplot(grid[0, 1]); ax.axis("off")
field_panel(ax, field, "routes from 14 cells")
for point in starts:
    route(ax, descend(point)[0], lw=1.2); dot(ax, point, ACCENT_GREEN, 22)
dot(ax, goal, ACCENT_RED)

# 3) d along one route
ax = fig.add_subplot(grid[0, 2])
ax.plot(height_along, color=rgb(ACCENT_BLUE), lw=2)
ax.axhline(cfg.reach, color=rgb(ACCENT_RED), lw=0.8, ls="--")
ax.scatter([0, len(path) - 1], [height_along[0], height_along[-1]], s=45, color=[rgb(ACCENT_GREEN), rgb(ACCENT_RED)], edgecolors=rgb(TEXT_PRIMARY), zorder=5)
ax.set_xlabel("step along the route"); ax.set_ylabel("d"); ax.set_title("d decreases along the route"); ax.set_box_aspect(1)
ax.spines[["top", "right"]].set_visible(False)

## 7 · Steering
The controller finds the lowest-$d$ cell in an $11\times11$ window and compares its heading with the direction to that cell:
$|\Delta\text{yaw}|>15^\circ$ turns, $>2^\circ$ moves forward with a small trim, otherwise it moves forward.

In [ ]:
row, column = map(int, path[len(path) // 2])  # agent cell halfway down the route
R, pad = cfg.radius, cfg.radius + 6
r0, r1, c0, c1 = max(0, row - pad), min(height, row + pad + 1), max(0, column - pad), min(width, column + pad + 1)

# 1) Lowest-distance cell, breaking ties in row-major order
window = field[row - R:row + R + 1, column - R:column + R + 1]                                # [2R+1, 2R+1]
target = np.array(np.unravel_index(window.argmin(), window.shape)) + (row - R, column - R)    # (row, column)
toward = (target - (row, column))[::-1] / np.linalg.norm(target - (row, column))              # unit (x, y)
agent = np.array([column, row])

def rotate(vector, degrees):
    c, s = np.cos(np.deg2rad(degrees)), np.sin(np.deg2rad(degrees))
    return np.array([[c, -s], [s, c]]) @ vector

# 2) Local search window and steering decisions
fig, axes = panels(4)
cells = plt.get_cmap("viridis")((1 - window / d_max).clip(0, 1))[..., :3]
cells[~np.isfinite(window)] = MAP_WALL
ax = axes[0, 0]
show(ax, cells, "lowest d in the 11×11 window", extent=(column - R - 0.5, column + R + 0.5, row + R + 0.5, row - R - 0.5), interpolation="nearest")
ticks = np.arange(-R - 0.5, R + 1.5)
ax.vlines(column + ticks, row - R - 0.5, row + R + 0.5, color=rgb(TRAIL_SHADOW), lw=0.4); ax.hlines(row + ticks, column - R - 0.5, column + R + 0.5, color=rgb(TRAIL_SHADOW), lw=0.4)
route(ax, path); dot(ax, target, TEXT_PRIMARY, 90, hollow=True); dot(ax, (row, column), ACCENT_BLUE, 90)

crop = field_image(field, d_max)[r0 * (cfg.size // height):r1 * (cfg.size // height), c0 * (cfg.size // height):c1 * (cfg.size // height)]
for ax, offset, title in zip(axes[0, 1:], (40, 8, 0), ("turn", "forward + trim", "forward")):
    show(ax, crop, f"Δyaw {offset}° · {title}", extent=(c0 - 0.5, c1 - 0.5, r1 - 0.5, r0 - 0.5))
    contours(ax, field, np.linspace(0, d_max, 60), linewidths=0.5, alpha=0.3)
    route(ax, path)
    ax.add_patch(Rectangle((column - R - 0.5, row - R - 0.5), 2 * R + 1, 2 * R + 1, fill=False, ec=rgb(TEXT_LABEL), lw=1))
    heading = rotate(toward, offset)
    heading_cone(ax, agent, heading)
    ax.plot([column, target[1]], [row, target[0]], color=rgb(TEXT_PRIMARY), lw=1, zorder=5)
    dot(ax, target, TEXT_PRIMARY, 60, hollow=True); dot(ax, (row, column), ACCENT_BLUE, 90)
    if offset:
        angles = sorted(np.degrees(np.arctan2(v[1], v[0])) for v in (heading, toward))  # data y points down, as does the arc
        ax.add_patch(Arc(agent, 9, 9, theta1=angles[0], theta2=angles[1], color=rgb(TEXT_PRIMARY), lw=1.2, zorder=5))

## 8 · Waypoints
One field per waypoint, re-solved when the agent enters the red contour ($d<26$) or stalls.

In [ ]:
fig, axes = panels(len(waypoints), size=2.7)
for i, (ax, point) in enumerate(zip(axes[0], waypoints)):
    field, _ = compute_distance_map(free, point)
    field_panel(ax, field, f"waypoint {i + 1}", d_max=field[np.isfinite(field)].max(), point=point)
    contours(ax, field, [cfg.reach], color=rgb(ACCENT_RED), linewidths=1.2)